# GEN-KNOB Tuner — MoE LoRA Inference

This notebook uses the Mixture-of-Experts (MixLoRA) fine-tuned adapter over the E2ETune Mistral 7B base model. 
It includes the same advanced prompt preprocessing (Hardware Spec Simplification, Query Plan Structured Encoding, and Human-Readable Metric Scaling) that was used during the MoE training.

## Instructions
1. Run the installation and setup cells.
2. Modify the `inference_payload` in **Cell 5** with your specific workload, metrics, query plans, and hardware settings.
3. Run the inference generation cells to get your recommended knob configurations.

In [ ]:
# 1. Install dependencies
!pip install -q -U transformers datasets peft bitsandbytes mixlora safetensors huggingface_hub

In [ ]:
# 2. Imports and Initialisation
import os, json, re, ast, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import mixlora # Must be imported to register MoE extensions with PEFT
from peft import PeftModel

print("Imports successful!")

In [ ]:
# 3. Advanced Prompt Preprocessing (Matching the Training Notebook)

def parse_hardware_spec(hw_raw: str) -> str:
    hw = str(hw_raw).lower()
    ram_match     = re.search(r"(\d+)\s*gb", hw)
    cores_match   = re.search(r"(\d+)\s*c(?:ore|pu)?(?:\b|[-_])", hw)
    threads_match = re.search(r"(\d+)\s*t(?:hread)?(?:\b|[-_])", hw)

    ram     = f"{ram_match.group(1)} GB"     if ram_match     else "unknown RAM"
    cores   = f"{cores_match.group(1)} cores"   if cores_match   else "unknown cores"
    threads = f"{threads_match.group(1)} threads" if threads_match else "unknown threads"

    return f"{ram} RAM, {cores}, {threads}"

def _extract_operator(node: dict) -> str:
    return node.get("Node Type", node.get("node_type", "Unknown"))

def _extract_cost(node: dict) -> float:
    return float(node.get("Total Cost", node.get("total_cost", node.get("cost", 0.0))))

def _flatten_plan_tree(node, operator_costs: dict):
    op   = _extract_operator(node)
    cost = _extract_cost(node)
    if op not in operator_costs:
        operator_costs[op] = [0.0, 0]
    operator_costs[op][0] += cost
    operator_costs[op][1] += 1
    for child_key in ("Plans", "plans", "children"):
        for child in node.get(child_key, []):
            _flatten_plan_tree(child, operator_costs)

def _parenthesise_str_plan(plan_str: str) -> str:
    pattern = re.compile(r"([A-Za-z][A-Za-z ]*?)\(cost=([\d.]+)\)")
    operator_costs: dict = {}
    for op, cost_str in pattern.findall(plan_str):
        op   = op.strip()
        cost = float(cost_str)
        if op not in operator_costs:
            operator_costs[op] = [0.0, 0]
        operator_costs[op][0] += cost
        operator_costs[op][1] += 1
    return _encode_operator_costs(operator_costs)

def _encode_operator_costs(operator_costs: dict) -> str:
    if not operator_costs:
        return "No plan"
    sorted_ops = sorted(
        operator_costs.items(),
        key=lambda kv: kv[1][0] / max(kv[1][1], 1),
        reverse=True
    )
    parts = []
    for op, (total, count) in sorted_ops:
        avg = total / max(count, 1)
        parts.append(f"{op}(cost={avg:.1f})")
    result = parts[0]
    for p in parts[1:]:
        result = f"{result}({p})"
    return result

def encode_query_plans(plans_raw) -> str:
    if not plans_raw:
        return "No query plans available."
    encoded = []
    for plan in plans_raw:
        if isinstance(plan, dict):
            op_costs: dict = {}
            _flatten_plan_tree(plan, op_costs)
            encoded.append(_encode_operator_costs(op_costs))
        elif isinstance(plan, str) and plan.strip():
            encoded.append(_parenthesise_str_plan(plan.strip()))
    return " | ".join(encoded) if encoded else "No query plans available."

def humanize_number(val) -> str:
    try:
        n = float(val)
    except (TypeError, ValueError):
        return str(val)
    abs_n = abs(n)
    sign  = "-" if n < 0 else ""
    if abs_n >= 1_000_000_000:
        return f"{sign}{abs_n / 1_000_000_000:.1f} billion"
    elif abs_n >= 1_000_000:
        return f"{sign}{abs_n / 1_000_000:.1f} million"
    elif abs_n >= 1_000:
        return f"{sign}{abs_n / 1_000:.1f} thousand"
    elif abs_n == 0:
        return "0"
    else:
        return f"{sign}{abs_n:.4g}"

def humanize_metrics(metrics: dict) -> dict:
    return {k: humanize_number(v) for k, v in metrics.items() if v != 0}

def format_mistral_prompt(payload: dict) -> str:
    db_name   = str(payload.get('database', 'UNKNOWN')).upper()
    hw_parsed = parse_hardware_spec(payload.get('hardware_specs', ''))

    raw_metrics  = payload.get('internal_metrics', {}) or {}
    metrics_str  = ", ".join(f"{k} = {v}" for k, v in humanize_metrics(raw_metrics).items())

    features     = payload.get('workload_features', {}) or {}
    features_str = ", ".join(f"{k} = {v}" for k, v in features.items())

    q_plan_str = encode_query_plans(payload.get('query_plans', []))

    instruction = (
        f"You are an expert {db_name} Database Administrator.\n"
        f"The server hardware is: {hw_parsed}.\n\n"
        "Given the internal system metrics, workload characteristics, and query execution plans below, "
        "recommend the optimal discrete bucket configuration for each database knob.\n\n"
        f"INTERNAL SYSTEM METRICS:\n{metrics_str}\n\n"
        f"WORKLOAD FEATURES:\n{features_str}\n\n"
        f"QUERY PLANS:\n{q_plan_str}"
    )

    return f"<s>[INST] {instruction} [/INST]\n"

print("Prompt preprocessing tools initialized!")

In [ ]:
# 4. Input Target Payload
import json
import os

INPUT_FILE = "/kaggle/input/datasets/nisith210144g/job-bm/job.json"
TARGET_DATABASE = "mysql"  # Switch to 'postgresql' if needed
TARGET_HARDWARE = "hetzner-4c-8t-32gb"

# Fallback for local workspace testing if Kaggle path isn't found
if not os.path.exists(INPUT_FILE) and os.path.exists("../inferencing/mysql/job/job.json"):
    INPUT_FILE = "../inferencing/mysql/job/job.json"
elif not os.path.exists(INPUT_FILE):
    INPUT_FILE = "job.json" # Generic fallback

print(f"Loading inference payload from {INPUT_FILE}...")
with open(INPUT_FILE, 'r') as f:
    inference_payload = json.load(f)

# The raw JSON structure typically only has metrics, plans, and features.
# We inject the target database and hardware specs if they are missing.
if "database" not in inference_payload:
    inference_payload["database"] = TARGET_DATABASE
if "hardware_specs" not in inference_payload:
    inference_payload["hardware_specs"] = TARGET_HARDWARE

formatted_prompt = format_mistral_prompt(inference_payload)
print("── Formatted Inference Prompt ──")
print(formatted_prompt)

In [ ]:
# 5. Load Base Model and MoE Adapter
import torch
import os
import json
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoConfig
from mixlora import MixLoraModelForCausalLM
from huggingface_hub import snapshot_download
from safetensors.torch import load_file

ADAPTER_NAME = "NisithDissanayake/genknob-tuner-moe"
KAGGLE_ADAPTER_DIR = "/kaggle/working/moe_adapter"
BASE_MODEL_ID = "springhxm/E2ETune"

print("Fetching and preparing adapter weights...")
local_dir = snapshot_download(ADAPTER_NAME, local_dir=KAGGLE_ADAPTER_DIR)
bin_path = os.path.join(local_dir, "adapter_model.bin")

# ── IMPORTANT HOTFIX FOR HUGGING FACE PUSH LOSS ──
# PEFT's push_to_hub strips out custom MixLoRA (MLP Expert & Router Gate) weights 
# and only uploads standard Attention LoRAs. We must dynamically rebuild the lost 
# MixLoRA skeleton here to avoid inference KeyErrors, then merge the surviving weights on top.
def rebuild_mixlora_skeleton(base_model_id, num_experts=2, rank=32):
    config = AutoConfig.from_pretrained(base_model_id)
    hidden_size = config.hidden_size
    num_layers = config.num_hidden_layers
    inter_size = config.intermediate_size
    num_heads = config.num_attention_heads
    kv_dim = getattr(config, "num_key_value_heads", num_heads) * (hidden_size // num_heads)

    dim_map = {
        "gate_proj": (hidden_size, inter_size),
        "up_proj": (hidden_size, inter_size),
        "down_proj": (inter_size, hidden_size)
    }

    state_dict = {}
    for i in range(num_layers):
        for e in range(num_experts):
            for proj in ["gate_proj", "up_proj", "down_proj"]:
                in_d, out_d = dim_map[proj]
                prefix = f"mixlora.layers.{i}.mlp.{proj}.experts.{e}"
                # Generate fallback dummy weights for the lost parameters
                state_dict[f"{prefix}.lora_A.weight"] = torch.randn((rank, in_d)) * 0.01
                state_dict[f"{prefix}.lora_B.weight"] = torch.zeros((out_d, rank))
        state_dict[f"mixlora.layers.{i}.mlp.moe_gate.weight"] = torch.randn((num_experts, hidden_size)) * 0.01
    return state_dict

print("Rebuilding missing MixLoRA topology...")
skeleton_weights = rebuild_mixlora_skeleton(BASE_MODEL_ID, num_experts=2, rank=32)

possible_safetensors = [
    os.path.join(local_dir, "adapter_model.safetensors"),
    os.path.join(local_dir, "model.safetensors")
]

if os.path.exists(bin_path):
    os.remove(bin_path)

for st_path in possible_safetensors:
    if os.path.exists(st_path):
        print(f"Found {os.path.basename(st_path)}. Merging with MixLoRA skeleton...")
        tensors = load_file(st_path)
        for k, v in tensors.items():
            if "lora_" in k or "moe_" in k:
                # Merge surviving weights (mostly Attention LoRAs)
                new_k = k.replace("base_model.model.model.layers.", "mixlora.layers.")
                new_k = new_k.replace("model.layers.", "mixlora.layers.")
                skeleton_weights[new_k] = v
        torch.save(skeleton_weights, bin_path)
        break

print("Loading 4-bit Base Model and MixLoRA Adapter...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model, config = MixLoraModelForCausalLM.from_pretrained(
    local_dir, 
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    use_cache=True
)
model.eval()

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("\nModel ready for inference!")

In [ ]:
# 6. Output Extraction Helper

def extract_json(s: str):
    m = re.search(r"\{.*?\}", s, flags=re.DOTALL)
    if not m: return None
    block = m.group(0)
    try: return json.loads(block)
    except Exception:
        try: return ast.literal_eval(block)
        except Exception: return None

In [ ]:
# 7. Generate Sequential Configurations
print("Generating 8 Diverse MoE Configurations...")
all_configs = []

inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

for i in range(8):
    print(f"\n[{i+1}/8] Generating...")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1024,
            temperature=0.7,
            do_sample=True,
            repetition_penalty=1.1,
            top_p=0.95,
            eos_token_id=tokenizer.eos_token_id
        )
    
    
    # Slice off the prompt to only decode the response
    prompt_length = inputs["input_ids"].shape[1]
    response_tokens = outputs[0][prompt_length:]
    
    text = tokenizer.decode(response_tokens, skip_special_tokens=True)
    
    pred_dict = extract_json(text)
    
    if pred_dict:
        all_configs.append(pred_dict)
        print(f"✓ Extracted successfully: {len(pred_dict)} knobs generated.")
        print(json.dumps(pred_dict, indent=2))
    else:
        print("Failed to extract JSON. Raw response:\n", text[:200])

output_file = "/kaggle/working/moe_generated_configs.json"
# Ensure directory exists (if running locally natively mapping to kaggle struct is weird)
os.makedirs(os.path.dirname(output_file), exist_ok=True) 

with open(output_file, 'w') as f:
    json.dump(all_configs, f, indent=2)

print(f"\nDone! ✓ Saved {len(all_configs)} MoE configurations to {output_file}")